**Install Required Libraries**

In [1]:
import sys
!{sys.executable} -m pip install flask pandas nltk transformers torch openpyxl --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


**Import Libraries & Initialize the AI Models**

In [2]:
import os
import nltk
from flask import Flask, render_template, request
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from transformers import pipeline

# Download the AI dictionary for sentiment tracking
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

# Load the advanced emotion analysis model
print("Loading AI models... please wait a moment.")
emotion_classifier = pipeline(
    "text-classification", 
    model="j-hartmann/emotion-english-distilroberta-base", 
    top_k=1
)
print("AI Models loaded successfully!")

c:\Users\Sunbal\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading AI models... please wait a moment.


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 20989.51it/s]


AI Models loaded successfully!


**Define the Analysis Functions**

In [3]:
def get_sentiment(text):
    if not text.strip():
        return "Neutral"
    scores = sia.polarity_scores(text)
    compound = scores['compound']
    if compound >= 0.05:
        return 'Positive'
    elif compound <= -0.05:
        return 'Negative'
    else:
        return 'Neutral'

def get_emotion(text):
    if not text.strip():
        return "neutral"
    try:
        result = emotion_classifier(text)[0]
        return result[0]['label']
    except Exception:
        return "unknown"

**Create the Webpage Layout (HTML)**

In [4]:
# Create the templates folder automatically
os.makedirs("templates", exist_ok=True)

# Define the HTML webpage structure
html_layout = """
<!DOCTYPE html>
<html>
<head>
    <title>Sentiment Analyzer App</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 50px; background-color: #f7f9fa; }
        .box { max-width: 500px; margin: auto; background: white; padding: 25px; border-radius: 8px; box-shadow: 0px 4px 10px rgba(0,0,0,0.1); }
        textarea { width: 100%; height: 100px; padding: 10px; box-sizing: border-box; font-size: 16px; border: 1px solid #ccc; border-radius: 4px; }
        button { width: 100%; background-color: #007bff; color: white; padding: 12px; border: none; font-size: 16px; border-radius: 4px; cursor: pointer; margin-top: 10px; }
        button:hover { background-color: #0056b3; }
        .result { margin-top: 20px; padding: 15px; border-radius: 4px; background-color: #e9ecef; border-left: 6px solid #6c757d; }
    </style>
</head>
<body>
    <div class="box">
        <h2>Sentiment & Emotion Web App</h2>
        <p>Type your text below to analyze it instantly in your browser:</p>
        <form method="POST">
            <textarea name="review_text" placeholder="Enter text here..." required>{{ text }}</textarea>
            <button type="submit">Analyze Sentiment</button>
        </form>
        
        {% if sentiment %}
        <div class="result">
            <h3>Results:</h3>
            <p><strong>Overall Sentiment:</strong> {{ sentiment }}</p>
            <p><strong>Detected Emotion:</strong> {{ emotion }}</p>
        </div>
        {% endif %}
    </div>
</body>
</html>
"""

# Save the file
with open("templates/index.html", "w") as f:
    f.write(html_layout)

print("Webpage template built successfully!")

Webpage template built successfully!


**Start the Server & Launch the Web App**

In [ ]:
app = Flask(__name__)

@app.route("/", methods=["GET", "POST"])
def index():
    sentiment = None
    emotion = None
    text = ""
    
    if request.method == "POST":
        text = request.form["review_text"]
        sentiment = get_sentiment(text)
        emotion = get_emotion(text)
        
    return render_template("index.html", sentiment=sentiment, emotion=emotion, text=text)

if __name__ == "__main__":
    app.run(host="127.0.0.1", port=5000, debug=False, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [12/Jul/2026 00:31:59] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Jul/2026 00:31:59] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [12/Jul/2026 00:32:32] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [12/Jul/2026 00:33:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Jul/2026 00:34:04] "POST / HTTP/1.1" 200 -
